# 🛡️ PharmaGuard — AI Pharmacy Security Monitor
### Real-time Suspicious-Interaction Monitoring System

This Google Colab notebook runs the cloud processing backend for **PharmaGuard**.
It ingests live webcam frames transmitted securely from your laptop browser, runs GPU-accelerated person detection (pretrained YOLO) and multi-object tracking (ByteTrack), analyzes pharmacy shelf dwell interactions, and streams annotated telemetry back to your browser in real time.

---

## Step 1: Verify Hardware & GPU Environment
Verify that an active NVIDIA GPU (T4, L4, A100, or V100) is allocated to this Colab runtime.

In [ ]:
!nvidia-smi

import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"Active GPU Device: {gpu_name} ({gpu_mem:.2f} GB VRAM)")
else:
    print("WARNING: Running on CPU. For real-time performance, select: Runtime > Change runtime type > T4 GPU.")

## Step 2: Mount Google Drive (Optional for Persistent Storage)
Persisting weights and alert logs on Google Drive keeps them safe across session restarts.

In [ ]:
import os
try:
    from google.colab import drive
    drive.mount('/content/drive')
    PERSISTENT_DIR = "/content/drive/MyDrive/PharmaGuard_Artifacts"
    os.makedirs(PERSISTENT_DIR, exist_ok=True)
    print(f"Google Drive persistent storage ready at: {PERSISTENT_DIR}")
except Exception as e:
    print(f"Skipping Google Drive mount: {e}. Running in local Colab disk.")

## Step 3: Setup PharmaGuard Workspace
If you uploaded the project directly or cloned it, navigate to the project directory.

In [ ]:
import os

# Check if running from repository directory or /content
if os.path.exists("/content/pharmguard"):
    %cd /content/pharmguard
elif os.path.exists("pharmaguard"):
    print(f"Already in project directory: {os.getcwd()}")
else:
    print("Cloning repository...")
    !git clone https://github.com/Franklinnn099/Pharmguard.git /content/pharmguard
    %cd /content/pharmguard

!pwd
!ls -la

## Step 4: Install Pinned Dependencies

In [ ]:
!pip install -q -r requirements.txt
!pip install -q --user lap || true

## Step 5: GPU Benchmark & Pipeline Verification
Warm up model weights on the GPU, run 20 test frames, and measure actual GPU inference latency.

In [ ]:
import time
import numpy as np
import torch
from pharmaguard.config import PharmaGuardConfig
from pharmaguard.pipeline import PharmaGuardPipeline

config = PharmaGuardConfig()
config.detector.device = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Initializing PharmaGuard pipeline on {config.detector.device.upper()}...")
pipeline = PharmaGuardPipeline(config)

# Warmup inference
dummy_frame = np.zeros((480, 640, 3), dtype=np.uint8)
pipeline.process_frame(dummy_frame)

# Benchmark 20 frames
latencies = []
for _ in range(20):
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    t0 = time.perf_counter()
    out = pipeline.process_frame(dummy_frame)
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    t1 = time.perf_counter()
    latencies.append((t1 - t0) * 1000.0)

avg_lat = np.mean(latencies)
print(f"=" * 50)
print(f"GPU Benchmark Complete:")
print(f"  - Device: {pipeline.detector.get_model_info().get('device').upper()}")
print(f"  - Mean Pipeline Latency: {avg_lat:.2f} ms")
print(f"  - Potential Throughput:  {1000.0 / avg_lat:.1f} FPS")
print(f"  - Stage breakdown:       {out.metadata.get('stage_timings')}")
print(f"=" * 50)

## Step 6: Launch Live Surveillance Console with Public HTTPS Tunnel
This runs PharmaGuard and generates a public **`https://xxxx.gradio.live`** link.

### Instructions for Live Demonstration:
1. Click the public **`https://xxxx.gradio.live`** link printed below.
2. On your laptop browser, click the **Webcam** input button on the left panel.
3. When prompted by your browser (`Allow access to your camera?`), select **Allow**.
4. Live webcam frames will transmit to this Colab GPU runtime and return fully annotated with bounding boxes, ByteTrack IDs, shelf dwell timers, and real-time alerts!

In [ ]:
# Launch with Colab preset (share=True)
!python main.py --colab --model yolov8n.pt